# LangGraph — From Basic Chatbot to Agent, Memory & Streaming

This notebook explains the LangGraph tutorial step by step.

We will build the concepts in this order:

1. **Basic chatbot:** `START → LLM → END`
2. **Chatbot + tools:** LLM can request a search/calculation tool
3. **ReAct-style agent:** `LLM → Tool → LLM` loop
4. **Memory/checkpointing:** conversation state using `MemorySaver` + `thread_id`
5. **Streaming:** `invoke()`, `stream()`, `updates`, `values`, and async events

> **Important:** This is not one single chatbot. It is a progression from a simple graph to an agentic graph.

### Core mental model

- **State** = information travelling through the graph
- **Node** = a Python function that performs work
- **Edge** = connection between nodes
- **Conditional edge** = decision about which node comes next
- **START** = graph entry
- **END** = graph exit
- **ToolNode** = prebuilt node that executes tool calls
- **Checkpointer** = saves graph state/checkpoints
- **thread_id** = identifies a conversation/thread

## 0. Install the packages

Run this only if these packages are not already installed.

You can install them from a notebook cell with:

```bash
%pip install -U langgraph langchain langchain-groq python-dotenv langchain-tavily
```

You will also need:

- `GROQ_API_KEY` for the Groq model
- `TAVILY_API_KEY` for the Tavily search tool

Put them in a `.env` file in your project folder:

```text
GROQ_API_KEY=your_key_here
TAVILY_API_KEY=your_key_here
```

Do not commit your `.env` file to GitHub.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# State is the information carried through the graph.
# `add_messages` tells LangGraph how to update the messages field:
# new messages are merged into the existing message history.

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

print("State created")

# 1. Connect LangGraph to an LLM

LangGraph does not replace the LLM.

We still use a LangChain chat model. LangGraph controls the **workflow around the model**.

Conceptually:

```text
Python
  ↓
LangChain Chat Model
  ↓
Groq
  ↓
Llama
```

The model can be changed later without changing the basic LangGraph concepts.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model

# Change the model name if your Groq account/tutorial uses another supported model.
llm = init_chat_model("groq:llama-3.1-8b-instant")

print(llm)

## 1.1 What is a Node?

A **Node** is just a Python function.

It receives the current graph `state`, does some work, and returns an update to the state.

For the basic chatbot, the node simply sends the conversation messages to the LLM.

In [ ]:
def chatbot(state: State):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }

### What happens inside `chatbot()`?

```text
State
  ↓
state["messages"]
  ↓
LLM
  ↓
AIMessage
  ↓
updated State
```

This line:

```python
llm.invoke(state["messages"])
```

calls the LLM.

This:

```python
return {"messages": [response]}
```

returns the new message to LangGraph.

Because the State uses `add_messages`, the new message is merged with the existing message history.

## 1.2 Build the basic graph

Our first graph is:

```text
START
  ↓
chatbot
  ↓
END
```

In [ ]:
graph_builder = StateGraph(State)

# Add a node.
graph_builder.add_node("chatbot", chatbot)

# Connect START → chatbot.
graph_builder.add_edge(START, "chatbot")

# Connect chatbot → END.
graph_builder.add_edge("chatbot", END)

# Compile the graph so it can be executed.
graph = graph_builder.compile()

print("Basic chatbot graph compiled.")

## 1.3 Visualize the graph

The graph should look approximately like:

```text
┌───────┐
│ START │
└───┬───┘
    ↓
┌─────────┐
│ chatbot │
└────┬────┘
     ↓
┌───────┐
│  END  │
└───────┘
```

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 1.4 Run the basic chatbot

`invoke()` runs the graph and returns the final state.

`[-1]` means "take the last message".

In [ ]:
from langchain_core.messages import HumanMessage

response = graph.invoke({
    "messages": [HumanMessage(content="Hi, what is LangGraph?")]
})

print(response["messages"][-1].content)

### Is this an agent?

**No.**

This is only a basic chatbot:

```text
User
 ↓
LLM
 ↓
Answer
```

There is:

- no tool
- no conditional routing
- no agent loop
- no checkpoint memory

Next we add tools.

# 2. Chatbot + Tools

Now we give the LLM access to two tools:

1. A web search tool
2. A custom Python `multiply()` function

The important idea is:

```text
LLM
 ↓
"Should I use a tool?"
 ↓
Tool call
```

`bind_tools()` makes the tool definitions available to the model.

It does **not** execute the tools itself.

In [ ]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=2)

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [search_tool, multiply]

llm_with_tools = llm.bind_tools(tools)

print("Tools bound to the model.")

## 2.1 Test the custom tool directly

Before involving LangGraph, understand the normal Python function.

In [ ]:
print(multiply(5, 10))

## 2.2 Test the search tool directly

This requires `TAVILY_API_KEY`.

If you do not have a Tavily key yet, skip this cell and continue with the `multiply` example.

In [ ]:
# Uncomment after setting TAVILY_API_KEY.

# result = search_tool.invoke({"query": "What is LangGraph?"})
# print(result)

## 2.3 Create an LLM node that can request tools

The node is almost the same as before.

The difference is that we call:

```python
llm_with_tools.invoke(...)
```

instead of:

```python
llm.invoke(...)
```

In [ ]:
def tool_calling_llm(state: State):
    response = llm_with_tools.invoke(state["messages"])

    return {
        "messages": [response]
    }

## 2.4 Add `ToolNode`

`ToolNode` is a prebuilt LangGraph node.

Its job is to execute the tool call requested by the LLM.

Conceptually:

```text
LLM says:
multiply(5, 10)
      ↓
  ToolNode
      ↓
multiply()
      ↓
50
```

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(State)

builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

## 2.5 Conditional edge

This is the first major difference from the basic chatbot.

We want:

```text
                  LLM
                   ↓
            Need a tool?
             /         \
           YES          NO
            ↓            ↓
          Tools         END
```

`tools_condition` checks whether the latest AI message contains a tool call.

In [ ]:
builder.add_edge(START, "tool_calling_llm")

builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)

# In this first version, the tool finishes the graph.
builder.add_edge("tools", END)

tool_graph = builder.compile()

display(Image(tool_graph.get_graph().draw_mermaid_png()))

## 2.6 Run a tool-calling example

Ask something that can be handled by the local Python tool.

The exact model behavior can vary, so inspect the messages if necessary.

In [ ]:
response = tool_graph.invoke({
    "messages": [HumanMessage(content="What is 5 multiplied by 10?")]
})

for message in response["messages"]:
    message.pretty_print()

### What did we build?

This is no longer just:

```text
START → LLM → END
```

It is:

```text
             LLM
              ↓
       Tool call needed?
          /         \
        YES          NO
         ↓            ↓
       TOOL          END
         ↓
        END
```

However, there is still **no loop back from the tool to the LLM**.

That is why we build the next version.

# 3. ReAct-style Agent

Now we add one important edge:

```text
Tools → LLM
```

This creates a loop.

The graph becomes:

```text
                 ┌──────────┐
                 │  START   │
                 └────┬─────┘
                      ↓
                 ┌─────────┐
                 │   LLM   │
                 └────┬────┘
                      ↓
               Tool call?
                /       \
              YES        NO
               ↓          ↓
             TOOL        END
               │
               └────────→ LLM
```

This is the basic structure behind a tool-using agent.

The loop allows the model to:

1. Decide
2. Call a tool
3. Receive the tool result
4. Decide again
5. Call another tool if needed
6. Eventually produce a final answer

In [ ]:
builder = StateGraph(State)

builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")

# Decide whether to go to tools or END.
builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition
)

# IMPORTANT: after a tool executes, go back to the LLM.
builder.add_edge("tools", "tool_calling_llm")

agent_graph = builder.compile()

display(Image(agent_graph.get_graph().draw_mermaid_png()))

## 3.1 ReAct mental model

ReAct is commonly explained as:

```text
Reason
  ↓
Act
  ↓
Observe
  ↓
Reason
  ↓
Act
  ↓
Observe
```

In this graph:

```text
LLM
 ↓
Tool
 ↓
LLM
 ↓
Tool
 ↓
LLM
 ↓
END
```

The LLM is responsible for deciding whether a tool is needed and which tool to request.

In [ ]:
response = agent_graph.invoke({
    "messages": [
        HumanMessage(content="What is 5 multiplied by 10?")
    ]
})

for message in response["messages"]:
    message.pretty_print()

## 3.2 Multiple tools

For a request such as:

> Find recent AI news and then multiply 5 by 10.

the agent may need more than one tool call.

A possible execution is:

```text
START
  ↓
LLM
  ↓
Search tool
  ↓
LLM
  ↓
multiply(5, 10)
  ↓
LLM
  ↓
END
```

The exact tool sequence is decided by the model, so the result can vary.

In [ ]:
# This requires TAVILY_API_KEY.
# Uncomment when Tavily is configured.

# response = agent_graph.invoke({
#     "messages": [
#         HumanMessage(
#             content="Find recent AI news and then multiply 5 by 10."
#         )
#     ]
# })
#
# for message in response["messages"]:
#     message.pretty_print()

# 4. Memory / Checkpointing

Now we add persistence to the graph execution.

The key components are:

```text
MemorySaver
     +
thread_id
     ↓
saved conversation state
```

A `thread_id` identifies which conversation/checkpoint history should be used.

For example:

```text
thread_id = "1" → Conversation A
thread_id = "2" → Conversation B
```

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

memory_graph = builder.compile(
    checkpointer=memory
)

## 4.1 Create a conversation thread

The configuration tells LangGraph which conversation/thread to use.

In [ ]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

response = memory_graph.invoke(
    {
        "messages": [
            HumanMessage(content="Hi, my name is Krish.")
        ]
    },
    config=config
)

print(response["messages"][-1].content)

## 4.2 Ask about the previous message

We use the **same `thread_id`**.

That allows the checkpointer to associate the new invocation with the same conversation.

In [ ]:
response = memory_graph.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config=config
)

print(response["messages"][-1].content)

### Important distinction

`MemorySaver` is a **checkpointer**.

It stores graph checkpoints associated with threads.

For production applications, you may use a persistent database-backed checkpointer instead of in-memory storage. `MemorySaver` is excellent for learning and local experiments.

# 5. Streaming

So far we used:

```python
graph.invoke(...)
```

Now we explore streaming.

### `invoke()`

Wait for the graph execution and return the result.

```text
START → nodes → END → final result
```

### `stream()`

Receive execution output as the graph progresses.

```text
START
 ↓
Node update
 ↓
Node update
 ↓
Node update
 ↓
END
```

In [ ]:
# A small graph specifically for demonstrating streaming.

def superbot(state: State):
    return {
        "messages": [
            llm.invoke(state["messages"])
        ]
    }

stream_builder = StateGraph(State)
stream_builder.add_node("SuperBot", superbot)
stream_builder.add_edge(START, "SuperBot")
stream_builder.add_edge("SuperBot", END)

stream_graph = stream_builder.compile()

display(Image(stream_graph.get_graph().draw_mermaid_png()))

## 5.1 `stream_mode="updates"`

`updates` focuses on what each node changed/returned.

Think:

```text
Node executed
     ↓
What did this node update?
```

In [ ]:
config = {
    "configurable": {
        "thread_id": "3"
    }
}

for chunk in stream_graph.stream(
    {
        "messages": [
            HumanMessage(
                content="Hi, my name is Krish and I like cricket."
            )
        ]
    },
    config=config,
    stream_mode="updates"
):
    print(chunk)

## 5.2 `stream_mode="values"`

`values` gives the current/full graph state after each step.

Think:

```text
updates → what changed?
values  → what is the state now?
```

In [ ]:
for chunk in stream_graph.stream(
    {
        "messages": [
            HumanMessage(
                content="Tell me briefly what LangGraph is."
            )
        ]
    },
    config=config,
    stream_mode="values"
):
    print(chunk)

## 5.3 Async streaming

`astream()` is the asynchronous version of streaming.

`astream_events()` gives detailed execution events.

This is useful when building asynchronous applications such as web APIs and real-time UIs.

In [ ]:
# Example syntax. Run inside an async environment.

async def demo_async_events():
    config = {
        "configurable": {
            "thread_id": "5"
        }
    }

    async for event in stream_graph.astream_events(
        {
            "messages": [
                HumanMessage(
                    content="Explain LangGraph in one sentence."
                )
            ]
        },
        config=config,
        version="v2"
    ):
        print(event)

# In a notebook you can run:
# await demo_async_events()

# 6. Final Summary — What You Built

Your original tutorial is actually **five lessons**.

## Lesson 1 — Basic chatbot

```text
START
  ↓
LLM
  ↓
END
```

Components:

- `State`
- `Node`
- `START`
- `END`
- `Edge`
- `compile()`
- `invoke()`

---

## Lesson 2 — Tool calling

```text
             LLM
              ↓
       Tool required?
        /         \
      YES          NO
       ↓            ↓
     TOOL          END
       ↓
      END
```

New components:

- `bind_tools()`
- `ToolNode`
- `tools_condition`
- conditional edges

---

## Lesson 3 — ReAct-style agent

```text
             LLM
              ↓
         Tool needed?
          /       \
        YES        NO
         ↓          ↓
       TOOL        END
         │
         └──────→ LLM
```

The important new concept is the **loop**.

---

## Lesson 4 — Memory

```text
Graph
  ↕
Checkpointer
  ↕
thread_id
```

New concepts:

- `MemorySaver`
- `checkpointer`
- `thread_id`

---

## Lesson 5 — Streaming

```text
invoke()
stream()
astream()
astream_events()
```

And:

```text
updates → what changed?
values  → what is the current state?
```

---

# 7. The Most Important LangGraph Concepts to Remember

| Component | Meaning |
|---|---|
| **State** | Data shared through the graph |
| **Node** | Python function that performs work |
| **Edge** | Connection from one node to another |
| **Conditional edge** | Chooses the next node based on a decision |
| **START** | Beginning of graph |
| **END** | End of graph |
| **Graph** | Complete workflow |
| **Reducer / `add_messages`** | Defines how State updates are merged |
| **ToolNode** | Executes tool calls |
| **`bind_tools()`** | Makes tools available to the model |
| **Checkpointer** | Saves graph checkpoints |
| **`thread_id`** | Identifies a conversation |
| **`invoke()`** | Execute and return result |
| **`stream()`** | Stream graph execution |
| **`updates`** | Node/state updates |
| **`values`** | Current/full state |

---

# 8. Final Mental Model

Think of LangGraph as a railway system:

```text
                    STATE
                      │
                      ▼
START ──→ NODE ──→ NODE ──→ NODE ──→ END
            │
            │
            ▼
       CONDITIONAL
        /        \
       /          \
    TOOL          END
       │
       └────────→ NODE
```

- **State** = cargo travelling through the railway
- **Node** = station where work happens
- **Edge** = railway track
- **Conditional edge** = junction that chooses a track
- **ToolNode** = station that executes a tool
- **Checkpointer** = stores the journey/state
- **thread_id** = identifies the journey/conversation
- **Graph** = the whole railway network

### Learning order

```text
State
  ↓
Node
  ↓
Edge
  ↓
START / END
  ↓
Conditional Edge
  ↓
ToolNode
  ↓
Agent Loop
  ↓
Memory / Checkpoint
  ↓
Streaming
```

This is the order I recommend you study rather than trying to memorize the whole original notebook at once.